
# 🟢 CHƯƠNG 1: NỀN TẢNG TOÁN HỌC & MẠNG NƠ-RON CƠ BẢN
---
### 1. Học Máy (Machine Learning) bản chất là gì?
Máy tính không "thông minh", nó chỉ là một cỗ máy dò dẫm tìm ra **Trọng số (Weights - $w$)** sao cho kết quả dự đoán khớp với thực tế nhất.
* **Công thức lõi:** $Prediction = w \cdot x$

### 2. Thuật toán Giảm độ dốc (Gradient Descent)
* **Sai số (Loss):** Đo lường xem máy đoán ngu cỡ nào. Hàm phổ biến: Mất mát bình phương (MSE) = $(Prediction - True\_Value)^2$.
* **Đạo hàm (Derivative):** Kim chỉ nam giúp máy tính biết nên tăng hay giảm trọng số $w$ để sai số nhỏ lại.
* **Cập nhật:** $w = w - (Learning\_Rate \cdot Derivative)$

### 3. Tính Phi Tuyến (Non-Linearity)
* Mọi sự phức tạp trên đời đều không phải là đường thẳng.
* **Hàm Kích Hoạt ReLU ($f(x) = \max(0, x)$):** Bẻ gãy các giá trị âm thành 0. Nó tạo ra các "nếp nhăn" cho bộ não AI, giúp AI học được các đường cong phức tạp. Không có ReLU, ghép 1000 lớp nơ-ron cũng chỉ bằng 1 lớp.

In [ ]:
# Mô phỏng AI học bằng Toán thuần túy (Không dùng thư viện)
x = 2.0; y_true = 6.0  # Tìm w sao cho w * 2 = 6
w = 1.0                # Đoán bừa w ban đầu
lr = 0.05              # Tốc độ học

print("--- CHƯƠNG 1: GRADIENT DESCENT THỦ CÔNG ---")
for step in range(10):
    prediction = w * x
    error = prediction - y_true
    loss = error ** 2
    derivative = 2 * error * x    # Đạo hàm
    w = w - lr * derivative       # Cập nhật trọng số
    
    if step % 2 == 0:
        print(f"Bước {step}: Trọng số w = {w:.3f} | Sai số: {loss:.3f}")
print(f"✅ AI đã tìm ra quy luật: w = {w:.0f}")

# 🔵 CHƯƠNG 2: BƯỚC TIẾN LÊN DEEP LEARNING VỚI PYTORCH

---
Việc tính đạo hàm bằng tay chỉ dành cho đồ chơi. Khi AI có hàng tỷ tham số (như GPT-4), con người không thể giải tích nổi.
* **Tại sao cần PyTorch?**
  1. **Tensors:** Chạy ma trận trên GPU siêu tốc.
  2. **Autograd:** Tự động tính toán MỌI đạo hàm trên đời chỉ với lệnh `loss.backward()`.
* **Bộ 3 Phép màu của Huấn luyện:**

  1. `optimizer.zero_grad()`: Xóa sạch bộ nhớ đạo hàm cũ.
  2. `loss.backward()`: Kích hoạt Autograd tính đạo hàm mới.
  3. `optimizer.step()`: Trừ đạo hàm vào Trọng số (Giống hệt vòng lặp thủ công ở trên).

In [ ]:
import torch
import torch.nn as nn

print("\n--- CHƯƠNG 2: PYTORCH AUTOGRAD ---")
# Dữ liệu dạng Tensor
X = torch.tensor([[3.0, 10.0]]) 
Y = torch.tensor([[500.0]])

# Mạng Nơ-ron Đa Tầng (Có ReLU ở giữa)
model = nn.Sequential(
    nn.Linear(2, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

for step in range(50):
    optimizer.zero_grad()            # 1. Reset
    pred = model(X)                  # 2. Đoán
    loss = criterion(pred, Y)        # 3. Tính sai số
    loss.backward()                  # 4. Tự động tính đạo hàm
    optimizer.step()                 # 5. Cập nhật

print(f"✅ PyTorch đã train xong! Dự đoán cuối: {model(X).item():.1f} (Giá thật: 500)")

# 🟣 CHƯƠNG 3: KIẾN TRÚC TRANSFORMER & CHATGPT

---
Làm sao AI hiểu được ngôn ngữ con người?
#### Bước 1: Mã hóa Từ vựng (Token Embeddings)
* Dịch chữ thành ID số (Tra từ điển).
* Ép ID số thành **Vector Không Gian** (`nn.Embedding`). Các từ cùng nghĩa sẽ có vector nằm gần nhau.

#### Bước 2: Mã hóa Vị trí (Positional Encoding)
* Mạng Transformer đọc tất cả các từ **cùng 1 lúc** (song song), nên nó bị "mù" thứ tự (không biết từ nào đứng trước).
* Phải cộng thêm một vector mang "số báo danh vị trí" vào vector từ vựng.
* $Input = Vector\_Ngữ\_Nghĩa + Vector\_Vị\_Trí$

#### Bước 3: Cơ chế Self-Attention (Tự Chú Ý - Trái tim của AI)
* Giúp các từ "nói chuyện" với nhau để tìm ngữ cảnh (vd: từ "Nó" là con mèo hay con chuột).
* Từng từ sinh ra 3 bản sao:
  - **Query (Q):** Tôi đang tìm kiếm điều gì?
  - **Key (K):** Tôi có nhãn dán/đặc tính gì?
  - **Value (V):** Giá trị ngữ nghĩa thực sự của tôi.
* **Công thức cốt lõi:** Nhân Q với K (Chuyển vị) để xem độ khớp. Biến thành % bằng Softmax. Sau đó nhân lấy thông tin từ V.

In [ ]:
import math
import torch
import torch.nn as nn

print("\n--- CHƯƠNG 3: SELF-ATTENTION TRONG TRANSFORMER ---")
# Giả sử ta có 3 từ, mỗi từ là vector 4 chiều. Đã được cộng Positional Encoding.
# Kích thước x: [1 câu, 3 từ, vector 4 chiều]
x = torch.randn(1, 3, 4) 

d_model = 4
# Tạo 3 Lớp sinh ra Q, K, V
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

Q = W_q(x)
K = W_k(x)
V = W_v(x)

# Ma thuật Attention: Q nhân với K chuyển vị (-2, -1)
scores = Q @ K.transpose(-2, -1) / math.sqrt(d_model)

# Softmax biến điểm số thành % sự tập trung
attention_weights = torch.softmax(scores, dim=-1)

# Nhân % tập trung với Value thực tế của các từ
output = attention_weights @ V

print("Hình dáng đầu ra (Shape):", output.shape)
print("✅ Vector của mỗi từ hiện đã hòa quyện với ngữ cảnh của các từ xung quanh!")


--- CHƯƠNG 3: SELF-ATTENTION TRONG TRANSFORMER ---
Hình dáng đầu ra (Shape): torch.Size([1, 3, 4])
✅ Vector của mỗi từ hiện đã hòa quyện với ngữ cảnh của các từ xung quanh!
================ CHÚC MỪNG BẠN ĐÃ TỐT NGHIỆP KIẾN THỨC NỀN! ================


# 🎭 CHƯƠNG 4: MẶT NẠ TIÊN TRI (CAUSAL MASK) - BÍ MẬT CỦA GPT

---

### 🔮 1. Vấn đề "Nhìn trộm tương lai"
* Kiến trúc Transformer gốc (của Google) cho phép tất cả các từ nhìn thấy nhau (cả quá khứ lẫn tương lai) để **Dịch thuật** cho chuẩn.
* Nhưng **GPT** (của OpenAI) lại sinh ra để **Sáng tác/Dự đoán** từ tiếp theo.
* Nếu đang đứng ở từ số 1 mà AI lại nhìn thấy trước từ số 2 và 3, nó sẽ "gian lận" và không học được cách suy luận.

### 🔺 2. Giải pháp: Ma trận Tam giác dưới (Lower Triangular Matrix)
* Các kỹ sư AI tạo ra một "mặt nạ" hình tam giác:
  - **Số 1 (Bật):** Chỉ cho phép nhìn chính mình và các từ đứng trước (Quá khứ).
  - **Số 0 (Tắt):** Bịt mắt hoàn toàn các từ đứng sau (Tương lai).

### 🧮 3. Tại sao lại thay số 0 bằng Âm Vô Cùng (-inf)?
* Trong toán học của mạng Nơ-ron, điểm số Attention sẽ được đưa qua hàm **Softmax** để biến thành %.
* Đặc tính kỳ diệu của Softmax: $Softmax(-\infty) = 0\%$.
* Vậy nên ta thay các số 0 trên mặt nạ thành $-\infty$, để ép AI dành chính xác **0% sự tập trung** cho tương lai!

In [2]:
import torch
import torch
import torch.nn as nn

print("--- CHƯƠNG 4: ĐEO MẶT NẠ CHO MA TRẬN ATTENTION ---")

# 1. BẢNG ĐIỂM ATTENTION THÔ (Ví dụ câu 3 từ: "Tôi", "thích", "AI")
# Kích thước: [3 từ đang chấm điểm chéo cho 3 từ]
raw_scores = torch.tensor([
    [10.0, 20.0, 30.0],  # Từ 1 "Tôi" đang nhìn thấy cả điểm của tương lai (20, 30)
    [15.0, 25.0, 10.0],  
    [12.0, 18.0, 20.0]   
])

print("\n1. Bảng điểm ban đầu (Gian lận nhìn trộm tương lai):")
print(raw_scores)

# 2. TẠO MẶT NẠ (Hàm torch.tril tạo ma trận tam giác dưới)
mask = torch.tril(torch.ones(3, 3))

print("\n2. Hình dáng mặt nạ (1: Được nhìn | 0: Bịt mắt):")
print(mask)

# 3. ĐEO MẶT NẠ (Chỗ nào là 0 thì đè float('-inf') vào)
masked_scores = raw_scores.masked_fill(mask == 0, float('-inf'))

print("\n3. Bảng điểm sau khi đeo mặt nạ (-inf = Bịt mắt):")
print(masked_scores)

# 4. CHUYỂN THÀNH XÁC SUẤT BẰNG SOFTMAX
attention_weights = torch.softmax(masked_scores, dim=-1)

print("\n4. Kết quả % sự tập trung (Softmax):")
print(attention_weights)
print("=> Dòng 1 (Từ 'Tôi') giờ chỉ tập trung 100% (1.0) vào bản thân nó. Không còn bị lộ đáp án tương lai!")

--- CHƯƠNG 4: ĐEO MẶT NẠ CHO MA TRẬN ATTENTION ---

1. Bảng điểm ban đầu (Gian lận nhìn trộm tương lai):
tensor([[10., 20., 30.],
        [15., 25., 10.],
        [12., 18., 20.]])

2. Hình dáng mặt nạ (1: Được nhìn | 0: Bịt mắt):
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

3. Bảng điểm sau khi đeo mặt nạ (-inf = Bịt mắt):
tensor([[10., -inf, -inf],
        [15., 25., -inf],
        [12., 18., 20.]])

4. Kết quả % sự tập trung (Softmax):
tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00],
        [4.5398e-05, 9.9995e-01, 0.0000e+00],
        [2.9539e-04, 1.1917e-01, 8.8054e-01]])
=> Dòng 1 (Từ 'Tôi') giờ chỉ tập trung 100% (1.0) vào bản thân nó. Không còn bị lộ đáp án tương lai!


# 🥞 CHƯƠNG 5: MẠNG SUY NGHĨ SÂU & CHUẨN HÓA (FEED-FORWARD & LAYER NORM)

---

### ⚖️ 1. Người giữ thăng bằng (Layer Normalization)
* Sau nhiều phép nhân ma trận ở Self-Attention, các con số có thể bị cộng dồn lên quá to (ví dụ: 100.0, 500.0) hoặc ép xuống quá nhỏ.
* Nếu để nguyên, mạng nơ-ron sẽ bị "đột quỵ" (lỗi Gradient Exploding/Vanishing).
* **Layer Normalization** là một "chiếc van an toàn". Nó tự động tính trung bình cộng và ép các con số của mỗi từ về lại trạng thái ổn định (thường xoay quanh số 0, từ -1 đến 1).

### 🧠 2. Mạng Truyền Thẳng (Feed-Forward Network - FFN)
* Đây là không gian riêng để mỗi từ tự "tiêu hóa" ngữ cảnh thu được sau buổi họp nhóm.
* Cấu trúc siêu quen thuộc (Trò đã code ở Day 32): `Linear -> ReLU -> Linear`.
* **Quy tắc 4X của Transformer:** 1. Phóng to vector lên gấp 4 lần để "nhìn" vấn đề ở nhiều góc độ (vd: 64 chiều -> 256 chiều).
  2. Đi qua hàm kích hoạt **ReLU** để loại bỏ các thông tin rác (bẻ số âm thành 0) và giữ lại những đặc trưng sâu sắc nhất.
  3. Thu nhỏ vector về lại kích thước ban đầu (256 chiều -> 64 chiều) để chuẩn bị cho các khối tiếp theo.

In [3]:
import torch
import torch.nn as nn

print("--- CHƯƠNG 5: CHUẨN HÓA VÀ SUY NGHĨ SÂU ---")

# Giả sử ta có vector của từ "Tôi" sau khi đã "họp nhóm" (Self-Attention) xong
# Kích thước: [1 câu, 1 từ, vector 4 chiều]
d_model = 4
x_after_attention = torch.tensor([[[15.0, -20.0, 100.0, -5.0]]])

print("\n1. ĐẦU VÀO SAU KHI HỌP NHÓM (ATTENTION):")
print("Vector thô, các con số nhảy múa lộn xộn (có số rất to là 100.0):")
print(x_after_attention)

# ==========================================
# CHUẨN HÓA LỚP (LAYER NORMALIZATION)
# ==========================================
norm_layer = nn.LayerNorm(d_model)
x_normalized = norm_layer(x_after_attention)

print("\n2. SAU KHI CHUẨN HÓA (LAYER NORM):")
print("Các con số đã được ép về dạng cân bằng (mượt mà và an toàn hơn):")
print(x_normalized)

# ==========================================
# MẠNG SUY NGHĨ SÂU (FEED-FORWARD NETWORK)
# ==========================================
# Phóng to x4 -> ReLU -> Thu nhỏ
ffn = nn.Sequential(
    nn.Linear(d_model, d_model * 4), # Phóng to góc nhìn (4 -> 16)
    nn.ReLU(),                       # Bẻ gãy tuyến tính (Tạo nếp nhăn não)
    nn.Linear(d_model * 4, d_model)  # Rút ra kết luận (16 -> 4)
)

final_output = ffn(x_normalized)

print("\n3. ĐẦU RA CUỐI CÙNG (SAU FFN):")
print(f"Hình dáng: {final_output.shape}")
print(final_output)
print("=> ✅ TỪ NÀY ĐÃ TỰ TIÊU HÓA XONG NGỮ CẢNH VÀ ĐƯA RA ĐƯỢC ĐẶC TRƯNG SÂU SẮC NHẤT!")


--- CHƯƠNG 5: CHUẨN HÓA VÀ SUY NGHĨ SÂU ---

1. ĐẦU VÀO SAU KHI HỌP NHÓM (ATTENTION):
Vector thô, các con số nhảy múa lộn xộn (có số rất to là 100.0):
tensor([[[ 15., -20., 100.,  -5.]]])

2. SAU KHI CHUẨN HÓA (LAYER NORM):
Các con số đã được ép về dạng cân bằng (mượt mà và an toàn hơn):
tensor([[[-0.1615, -0.9152,  1.6690, -0.5922]]],
       grad_fn=<NativeLayerNormBackward0>)

3. ĐẦU RA CUỐI CÙNG (SAU FFN):
Hình dáng: torch.Size([1, 1, 4])
tensor([[[ 0.4232, -0.1379,  0.2213,  0.5326]]], grad_fn=<ViewBackward0>)
=> ✅ TỪ NÀY ĐÃ TỰ TIÊU HÓA XONG NGỮ CẢNH VÀ ĐƯA RA ĐƯỢC ĐẶC TRƯNG SÂU SẮC NHẤT!
